# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [14]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    # VS Code inyecta esta variable con la ruta absoluta del propio notebook,
    # así que la raíz del proyecto queda anclada a dónde vive el archivo .ipynb,
    # sin importar cuál sea el directorio de trabajo con el que arrancó el kernel
    # (que puede no ser la raíz del proyecto, según la configuración del editor).
    RAIZ_PROYECTO = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    RAIZ_PROYECTO = Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from modules.presentacion import (
    aplicar_tema_oscuro_notebook,
    ejecutar_extraccion_indice,
    exportar_csv_excel,
    exportar_ficha_a_pdf,
    exportar_xlsx,
    mostrar_emisoras,
    mostrar_ficha_completa_cliente,
    mostrar_ficha_rendimiento,
    probar_historial_dividendos,
    seleccionar_anio_interactivo,
    seleccionar_ticker_interactivo,
)
from modules.procesamiento import obtener_anios_disponibles

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
aplicar_tema_oscuro_notebook()

In [16]:
CARPETA_SALIDA = Path.cwd() / "output"
CARPETA_FICHAS_PDF = CARPETA_SALIDA / "fichas"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción
Consultar solo los lunes temprano para hacer un análisis rápido de las FIBRAS y para saber si hubo altas y bajas de emisoras.

In [ ]:
df = ejecutar_extraccion_indice(HEADLESS, TIMEOUT_DATOS_MS, CARPETA_SALIDA, EXPORTAR_CSV_ANALITICO)

### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [ ]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Consulta de emisoras

In [17]:
try:
    df
except NameError:
    df = None

df_emisoras = mostrar_emisoras(df, CARPETA_SALIDA)

Fuente de emisoras: extracción de AMEFIBRA de esta corrida.
      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18
CSV de emisoras guardado en: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_134450_list_of_tickers.csv


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

### Ticker a consultar

Elige, del desplegable, el ticker a consultar (mismo listado de la sección "Consula de emisoras"). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para consultar el ticker elegido.

In [18]:
selector_ticker = seleccionar_ticker_interactivo(df_emisoras["Emisora"])

Dropdown(description='Ticker:', options=('DANHOS13', 'EDUCA18', 'FIBRAMQ12', 'FIBRAPL14', 'FIBRAUP18', 'FIHO12…

In [19]:
TICKER_SELECCIONADO = selector_ticker.value
historial_dividendos = probar_historial_dividendos(TICKER_SELECCIONADO, df_emisoras["Emisora"], CARPETA_SALIDA)

Ticker probado: FIBRAPL14. Registros: 51
Periodicidad detectada: trimestral
CSV generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_134512_FIBRAPL14_dividendos.csv


,ticker,ex_date,amount_mxn,close_on_ex_date_mxn,yield_pct,annualized_yield_pct,periodicity
41,FIBRAPL14,2024-04-29,0.589204,70.510002,0.835632,5.446529,trimestral
42,FIBRAPL14,2024-08-19,0.682792,68.750000,0.993152,3.236611,trimestral
43,FIBRAPL14,2024-10-31,0.705053,67.000000,1.052318,5.261590,trimestral
44,FIBRAPL14,2025-02-06,0.721215,62.279999,1.158020,4.313035,trimestral
45,FIBRAPL14,2025-05-12,0.734250,69.879997,1.050730,4.037015,trimestral
46,FIBRAPL14,2025-08-11,0.695794,70.379997,0.988625,3.965363,trimestral
47,FIBRAPL14,2025-12-10,0.691400,72.029999,0.959878,2.895499,trimestral
48,FIBRAPL14,2026-02-13,0.646199,84.019997,0.769101,4.318800,trimestral
49,FIBRAPL14,2026-02-24,0.756600,81.389999,0.929598,30.845760,trimestral
50,FIBRAPL14,2026-08-03,0.731395,77.480003,0.943979,2.153452,trimestral


## Ficha de rendimiento anual

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

### Año a consultar

Elige, del desplegable, el año a consultar (solo se muestran los años con distribuciones disponibles para el ticker). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para generar la ficha con el año elegido.

In [20]:
AÑOS_DISPONIBLES = obtener_anios_disponibles(historial_dividendos)
selector_anio = seleccionar_anio_interactivo(AÑOS_DISPONIBLES)

Hay información disponible de 2014 a 2026.


Dropdown(description='Año:', options=(2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, …

In [21]:
# Generamos la ficha de rendimiento anual y la exportamos a PDF
AÑO_SELECCIONADO = selector_anio.value
ruta_ficha = mostrar_ficha_rendimiento(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_rendimiento = exportar_ficha_a_pdf(
    ruta_ficha, TICKER_SELECCIONADO, "rendimiento anual", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_rendimiento}")

Ficha generada: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_134536_FIBRAPL14_2025_ficha_rendimiento.html


ex_date,amount_mxn,yield_pct
2025-02-06,$0.7212,1.16%
2025-05-12,$0.7342,1.05%
2025-08-11,$0.6958,0.99%
2025-12-10,$0.6914,0.96%


PDF generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-30_1345_FIBRAPL14_rendimiento-anual_2025.pdf


### Ficha completa del año seleccionado para cliente

Ficha completa anual pensada como entregable final para el cliente (escenario de inversión, distribuciones mensuales y rendimiento total en el año), con un diseño distinto al de la ficha de rendimiento anterior. Usa el mismo ticker y año ya elegidos arriba y los mismos datos reales (`historial_dividendos`); no inventa cifras. Es informativa y no constituye una recomendación de inversión.

In [22]:
# Generamos la ficha completa anual para el cliente y la exportamos a PDF
ruta_ficha_completa_cliente = mostrar_ficha_completa_cliente(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_completa_cliente = exportar_ficha_a_pdf(
    ruta_ficha_completa_cliente, TICKER_SELECCIONADO, "ficha completa cliente", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_completa_cliente}")

Ficha completa generada: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_134554_FIBRAPL14_2025_ficha_completa_cliente.html


PDF generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-30_1345_FIBRAPL14_ficha-completa-cliente_2025.pdf
